# Project interactive visualization
Using a dataset that your group is consider using for the term project, let's build some meaningful user-driven data visualization. Depending on your dataset this could include:
- Usage of geospatial information
- Building interactive views with widgets
- Organize multiple components into a simple dashboard

Keep these design principles in mind:
While you navigate through this notebook some things to take into consideration:
* Do not add interaction just to add it. Make sure it helps answer a question.
* Use meaningful titles and labels
* Document your code so it's readable and clean. If something does not work, document the issue and explain your best attempt.

In [ ]:
## These are most likely the libraries you will use
# Add or remove imports as needed
!pip install jupyter_bokeh

import pandas as pd
import numpy as np

from plotly.offline import init_notebook_mode, iplot, plot
import plotly as py
init_notebook_mode(connected=True)
# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go

# Geospatial / geocoding
from geopy.geocoders import Nominatim

# Panel
import panel as pn
pn.extension('plotly')

!pip install hvplot
import hvplot.pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.6/180.6 kB 12.0 MB/s eta 0:00:00


In [ ]:
### If running plotly in Google colab, you will need to run this custom initialization function
### in each offline plotting cell to embed HTML content in Jupyter notebook
def configure_plotly_browser_state():
  import IPython
  display(IPython.core.display.HTML('''
        <script src="/static/components/requirejs/require.js"></script>
        <script>
          requirejs.config({
            paths: {
              base: '/static/base',
              plotly: 'https://cdn.plot.ly/plotly-latest.min.js?noext',
            },
          });
        </script>
        '''))

###Below is importing data and cleaning it, same as what we did in P2

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json
import glob
import os
import re
%matplotlib inline

!git clone https://github.com/HardwareDealsCo/ram-deals.git
#found this dataset on github and so we cloned it to this project file.

# Data is separated into different json files in the github, we want to combine to make one large dataset
files = glob.glob("ram-deals/historical-ram-deals-data/*.json")
#there are multiple json files in the original github dataset because they updated prices for each day

df_list = []

for fname in files:
    base = os.path.basename(fname)

    #I'm find the data pattern from the file name
    match = re.search(r"\d{4}-\d{2}-\d{2}", base)

    if match:
        date = pd.to_datetime(match.group())
    else:
        continue

    with open(fname) as f:
        data = json.load(f)

    temp_df = pd.DataFrame(data)

    # Add datetime column
    temp_df["date"] = date

    df_list.append(temp_df)

ram_data = pd.concat(df_list, ignore_index=True)

#too many values, we'll start with 30k random values to keep loading times down
ram_data = ram_data.sample(n=30000, random_state = 50)

ram_data.drop('itemGroupType', axis=1, inplace=True)

ram_data['price'] = pd.to_numeric(ram_data['price'], errors='coerce')

ram_data.drop(columns=["source", "currency", "link"], axis=1, inplace=True)

ram_data = ram_data[(ram_data.price <= 6000)]

ram_data = ram_data.dropna() #it's just one value, so we will drop this one na value, won't affect much since the df samples 30000 values

Cloning into 'ram-deals'...
remote: Enumerating objects: 1239, done.
remote: Total 1239 (delta 0), reused 0 (delta 0), pack-reused 1239 (from 3)
Receiving objects: 100% (1239/1239), 15.52 MiB | 26.32 MiB/s, done.
Resolving deltas: 100% (811/811), done.


In [ ]:
#defining own function to make the speed category simpler, either its DDR3, DDR4, or DDR5, instead of the 100+ different categories which all mean similar things
import re

def reclassify_speed(val):
  if not isinstance(val,  str):
    return None

  speed = val.lower()
  #start with the explicit definitions
  if "ddr3" in speed:
    return "DDR3"
  if "ddr4" in speed:
    return "DDR4"
  if "ddr5" in speed:
    return "DDR5"

  #there was no explicit ddr generation given, so we'll generally categorize into those 3 categories based on speed (educated guess)
  numerical_speed = re.search(r"\d+", speed)

  if not numerical_speed:
    return None

  speed = int(numerical_speed.group())
  if speed <= 2133:
      return "DDR3"
  elif 2133 < speed < 4800:
      return "DDR4"
  elif speed >= 4800:
      return "DDR5"

  return None

In [ ]:
ram_data.speed = ram_data.speed.apply(lambda x: reclassify_speed(x))
#going to rename from speed to generation since that name fits better
ram_data = ram_data.rename(columns={"speed": "generation"})

In [ ]:
# Write your code here
ram_data.head()

,title,brand,price,$/gb,capacity_gb,generation,form_factor,condition,date
307357,Dell Memory SNPX830DC/4G 4GB 2Rx8 DDR3 SODIMM ...,Dell,42.95,10.737500,4,DDR3,Laptop,New,2025-10-16
599307,G. SKILL 4 GB UDIMM 1600 MHz PC3-12800 DDR3 SD...,G.SKILL,17.99,4.497500,4,DDR3,Desktop,Used,2026-04-13
116184,Hynix 4 GB PC3-8500u DDR3-1066 240pin 1066 MHz...,SK Hynix,13.99,3.497500,4,DDR3,Desktop,New,2025-12-22
444262,Kingston FURY 64GB KIT DDR5 6400 RGB Memory DI...,Kingston,649.99,10.156094,64,DDR5,Desktop,Open box,2026-02-01
583798,SAMSUNG 16GB DDR3 1600 2Rx4 PC3L-12800R M393B2...,Samsung,100.00,6.250000,16,DDR3,Server,Used,2025-11-22


#Machine Learning Part

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

######Since the data set isn't missing any values, we don't need to impute. We will just stratify split using a custom "price_category" column, and use that to train the data

In [ ]:
#don't need the title category for the machine learning part
ml_ram_data = ram_data.drop("title", axis=1)

#turn the date into a numerical category
ml_ram_data["date"] = pd.to_datetime(ml_ram_data["date"]).astype("int64")

In [ ]:
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=50)

ml_ram_data["price_category"] = pd.cut(ml_ram_data["price"],
                               bins=[0, 500, 1000, 1500, 2000, np.inf],
                               labels=[1, 2, 3, 4, 5])

for train_index, test_index in split.split(ml_ram_data, ml_ram_data["price_category"]):
    strat_train_set = ml_ram_data.iloc[train_index]
    strat_test_set = ml_ram_data.iloc[test_index]

#remove the temporary classification for stratification
for set_ in (strat_train_set, strat_test_set):
    set_.drop("price_category", axis=1, inplace=True)

ram_training_set = strat_train_set.drop("price", axis=1)
ram_training_labels = strat_train_set["price"].copy()

ram_test_set = strat_test_set.drop("price", axis=1)
ram_test_labels = strat_test_set["price"].copy()

/tmp/ipykernel_25601/792216841.py:13: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_25601/792216841.py:13: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
ram_num = ram_training_set.drop(["brand", "generation", "form_factor", "condition"], axis=1)
ram_cat = ram_training_set.drop(["$/gb", "capacity_gb", "date"], axis=1)

cat_encoder = OneHotEncoder()

#get numerical values for the categorical parts using a one hot encoder
ram_cat_onehot = cat_encoder.fit_transform(ram_cat)

In [ ]:
numerical_pipeline = Pipeline([
        ('std_scaler', StandardScaler())
    ])

num_attribs = list(ram_num)
cat_attribs = list(ram_cat)

complete_pipeline = ColumnTransformer([
        ("num", numerical_pipeline, num_attribs),
        ("cat", OneHotEncoder(), cat_attribs),
    ])

ml_ram_prepared = complete_pipeline.fit_transform(ram_training_set)
ml_ram_prepared_test = complete_pipeline.transform(ram_test_set)

#was having issues with the class example, so got columns using this function
cols=complete_pipeline.get_feature_names_out()

ml_ram_prep_df = pd.DataFrame(
    ml_ram_prepared.toarray(),
    columns=cols,
    index=ram_training_set.index)

###Linear Regression Machine Learning Model (degree 2)

In [ ]:
polynomial = PolynomialFeatures(degree=2, include_bias = False) #playing around with polynomial regression
poly_ram_prepared = polynomial.fit_transform(ml_ram_prepared) #change the prepared data to a form that works with polynomial regression
poly_ram_prepared_test = polynomial.transform(ml_ram_prepared_test)

lin_reg = LinearRegression() #linear regression model that fits to the data in polynomial form
lin_reg.fit(poly_ram_prepared, ram_training_labels)



LinearRegression()

In [ ]:
ram_training_labels.describe()

,price
count,23988.000000
mean,192.227591
std,273.936377
min,1.500000
25%,31.950000
50%,96.950000
75%,268.925000
max,5199.990000


In [ ]:
lin_ram_predictions = lin_reg.predict(poly_ram_prepared_test)

lin_mse = mean_squared_error(ram_test_labels, lin_ram_predictions)
lin_rmse = np.sqrt(lin_mse)

lin_mae = mean_absolute_error(ram_test_labels, lin_ram_predictions)

lin_scores = cross_val_score(lin_reg, poly_ram_prepared, ram_training_labels,
                        scoring="neg_mean_squared_error", cv=10)
lin_rmse_scores = np.sqrt(-lin_scores)

print("cross validation mean (rmse): ", lin_rmse_scores.mean())
print("lin_rmse:", lin_rmse)
print("lin_mae:", lin_mae)


cross validation mean (rmse):  48.4726919339665
lin_rmse: 4.531059100174376
lin_mae: 2.5097771739567074


##### There was a very low error score when predicting on the test sets, with RMSE = 4.53 and MAE = 2.50. However, when looking at the cross validation scores on the training set with the RMSE metric, the error score comes out to be 48.47. This indicates there may be some overfitting with this particular model. However, it performs decently well, since it is a small portion of the standard deviation of 273.93.

###Decision Tree Regressor Machine Learning Model

In [ ]:
tree_reg = DecisionTreeRegressor(random_state=50)
tree_reg.fit(ml_ram_prepared, ram_training_labels)

DecisionTreeRegressor(random_state=50)

In [ ]:
dec_ram_predictions = tree_reg.predict(ml_ram_prepared_test)

dec_mse = mean_squared_error(ram_test_labels, dec_ram_predictions)
dec_rmse = np.sqrt(dec_mse) #root, mse and rmse are just different standards for finding errors

dec_mae = mean_absolute_error(ram_test_labels, dec_ram_predictions)

dec_scores = cross_val_score(tree_reg, ml_ram_prepared, ram_training_labels,
                        scoring="neg_mean_squared_error", cv=10)
dec_rmse_scores = np.sqrt(-dec_scores)

print("cross validation mean (rsme): ", dec_rmse_scores.mean()) #important
print("dec_rmse:", dec_rmse)
print("dec_mae:", dec_mae)



cross validation mean (rsme):  34.404162471180925
dec_rmse: 59.00542012634861
dec_mae: 2.90204336947459


#####When predicting on the test sets, MAE showed low error with an error margin of 2.90. However, RMSE showed higher error at 59.00. The high RMSE with the low MAE indicates that the model might be making a small number of mistakes with predicting the outliers, but at the same time is making pretty good predictions in general (excluding the outlieres).

The cross validation error score using the RMSE metric on the training set has a mean error score of 34.40. Again, although the model does not perform perfectly, it definitely is an improvement from the polynomial linear regression model. Again, this score is only a small fraction of the standard deviation (12%).

###Random Forest Regressor

In [ ]:
params = [
    {'n_estimators': [50, 100], 'max_features': [3, 7]},
    {'bootstrap': [False], 'n_estimators': [50, 100], 'max_features': [3,7]},
  ]

for_reg = RandomForestRegressor(random_state = 50)
grid_search = GridSearchCV(for_reg, params, cv=5,
                           scoring='neg_mean_squared_error', return_train_score=True)
grid_search.fit(ml_ram_prepared, ram_training_labels)

GridSearchCV(cv=5, estimator=RandomForestRegressor(random_state=50),
             param_grid=[{'max_features': [3, 7], 'n_estimators': [50, 100]},
                         {'bootstrap': [False], 'max_features': [3, 7],
                          'n_estimators': [50, 100]}],
             return_train_score=True, scoring='neg_mean_squared_error')

In [ ]:
forest_best_score = np.sqrt(-grid_search.best_score_)

print(forest_best_score.mean())

72.67435921351431
